<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:15px; color:white; margin:0; font-size:150%; font-family:Pacifico; background-color:#8B0000; overflow:hidden"><b> BERT Fine-tuning for Text Classification</b></div>

## BERT Fine-tuning for Text Classification
A complete, production-ready implementation of BERT fine-tuning using Hugging Face Transformers.
### 📋 Project Overview
This project demonstrates how to fine-tune a pre-trained BERT model for text classification tasks. It includes:

✅ Complete training pipeline

✅ Automatic GPU/CPU detection

✅ Comprehensive evaluation metrics

✅ Clean, documented code

✅ Production-ready architecture

### if you find this helpful please consider upvoting

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Instaling required libraries </b></div>

In [12]:
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Initialize Bert Transformer </b></div>

In [13]:
class BERTFineTuner:

    
    def __init__(self, model_name="bert-base-uncased", num_labels=2, max_length=128):

        self.model_name = model_name
        self.num_labels = num_labels
        self.max_length = max_length
        
       
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        
  
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels
        )
        self.model.to(self.device)
        
        print(f"Model loaded: {model_name}")
        print(f"Number of parameters: {sum(p.numel() for p in self.model.parameters()):,}")
    
    def load_and_prepare_data(self, dataset_name="imdb", sample_size=None):

        print(f"\nLoading dataset: {dataset_name}")
        
  
        dataset = load_dataset(dataset_name)
        

        if sample_size:
            dataset['train'] = Dataset.from_dict(dataset['train'][:sample_size])
            dataset['test'] = Dataset.from_dict(dataset['test'][:sample_size])
        
        print(f"Train samples: {len(dataset['train'])}")
        print(f"Test samples: {len(dataset['test'])}")
        
        # Tokenize the datasets
        train_dataset = self._tokenize_dataset(dataset['train'])
        test_dataset = self._tokenize_dataset(dataset['test'])
        
        return train_dataset, test_dataset
    
    def _tokenize_dataset(self, dataset):

        def tokenize_function(examples):
            
            # Tokenize the text
            tokenized = self.tokenizer(
                examples['text'],
                padding='max_length',  # Pad to max_length
                truncation=True,        # Truncate sequences longer than max_length
                max_length=self.max_length,
                return_tensors=None     # Return lists, not tensors (for batching)
            )
            
         
            if 'label' in examples:
                tokenized['labels'] = examples['label']
            
            return tokenized
        

        tokenized_dataset = dataset.map(
            tokenize_function,
            batched=True,
            remove_columns=dataset.column_names  # Remove original text columns
        )
        
        # Set format to PyTorch tensors
        tokenized_dataset.set_format('torch')
        
        return tokenized_dataset
    
    def compute_metrics(self, eval_pred):

        # Extract predictions and labels
        logits, labels = eval_pred
        
        # Get predicted class (argmax of logits)
        predictions = np.argmax(logits, axis=-1)
        
        # Calculate metrics
        accuracy = accuracy_score(labels, predictions)
        
        # F1-score with different averaging strategies
        f1_macro = f1_score(labels, predictions, average='macro') 
        f1_weighted = f1_score(labels, predictions, average='weighted') 
        
        return {
            'accuracy': accuracy,
            'f1_macro': f1_macro,
            'f1_weighted': f1_weighted
        }
    
    def train(self, train_dataset, test_dataset, output_dir="./results", 
              num_epochs=7, batch_size=16, learning_rate=2e-5):


        print("Starting Training")

        

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            
            # Optimizer settings
            learning_rate=learning_rate,
            weight_decay=0.01,  # L2 regularization
            
            # Evaluation strategy
            eval_strategy="epoch",  
            save_strategy="epoch",   
            load_best_model_at_end=True,  
            
            # Logging
            logging_dir=f'{output_dir}/logs',
            logging_steps=100,
            
            # Performance optimizations
            fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
            dataloader_num_workers=4 if torch.cuda.is_available() else 0,
            
            # Reproducibility
            seed=42,
            
            # Disable wandb/mlflow if not configured
            report_to="none"
        )
        

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        
        # Initialize Trainer
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=test_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics
        )
        

        print("\nTraining in progress...")
        train_result = trainer.train()
        
        # Save the final model
        trainer.save_model(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        

        print("Training Completed!")

        print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
        print(f"Samples per second: {train_result.metrics['train_samples_per_second']:.2f}")
        
        return trainer
    
    def evaluate(self, trainer, test_dataset):


        print("Evaluating Model")

        
        # Run evaluation
        eval_results = trainer.evaluate(test_dataset)
        
        # Print results
        print("\nEvaluation Results:")
        print("-" * 40)
        for metric, value in eval_results.items():
            if isinstance(value, float):
                print(f"{metric}: {value:.4f}")
            else:
                print(f"{metric}: {value}")
        
        return eval_results
    
    def predict(self, texts):

        # Tokenize input texts
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Move to device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Make predictions
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            
        # Get predicted classes and probabilities
        probabilities = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)
        
        return predictions.cpu().numpy(), probabilities.cpu().numpy()


<a id="1"></a>
# <div style="text-align:center; border-radius:15px 50px; padding:7px; color:white; margin:0; font-size:110%; font-family:Pacifico; background-color:#3168a1; overflow:hidden"><b> Main Run </b></div>

In [15]:
def main():

    print("BERT Fine-tuning for Text Classification")

    

    finetuner = BERTFineTuner(
        model_name="bert-base-uncased",
        num_labels=2,                     
        max_length=128                    
    )
    

    train_dataset, test_dataset = finetuner.load_and_prepare_data(
        dataset_name="imdb",
          
    )
    

    trainer = finetuner.train(
        train_dataset=train_dataset,
        test_dataset=test_dataset,
        output_dir="./bert_finetuned",
        num_epochs=5,              
        batch_size=16,             
        learning_rate=2e-5         
    )
    

    eval_results = finetuner.evaluate(trainer, test_dataset)
    

    print("Testing Predictions on New Samples")

    
    test_samples = [
        "This movie was absolutely fantastic! I loved every minute of it.",
        "Terrible film. Complete waste of time and money.",
        "It was okay, nothing special but not bad either."
    ]
    
    predictions, probabilities = finetuner.predict(test_samples)
    
    label_names = {0: "Negative", 1: "Positive"}
    
    for i, text in enumerate(test_samples):
        pred_label = label_names[predictions[i]]
        confidence = probabilities[i][predictions[i]] * 100
        
        print(f"\nText: {text}")
        print(f"Prediction: {pred_label} (Confidence: {confidence:.2f}%)")
        print(f"Probabilities: Negative={probabilities[i][0]:.4f}, Positive={probabilities[i][1]:.4f}")
    

    print("Fine-tuning Complete!")



if __name__ == "__main__":
    main()


BERT Fine-tuning for Text Classification
Using device: cuda


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: bert-base-uncased
Number of parameters: 109,483,778

Loading dataset: imdb
Train samples: 25000
Test samples: 25000


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.298200,0.271137,0.886400,0.886398,0.886398
2,0.207600,0.289037,0.888800,0.888736,0.888736
3,0.117100,0.404700,0.880520,0.880075,0.880075
4,0.061400,0.496558,0.888520,0.888511,0.888511
5,0.036000,0.539563,0.889560,0.889560,0.889560


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


Training Completed!
Training time: 2426.78 seconds
Samples per second: 51.51
Evaluating Model


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 


Evaluation Results:
----------------------------------------
eval_loss: 0.2711
eval_accuracy: 0.8864
eval_f1_macro: 0.8864
eval_f1_weighted: 0.8864
eval_runtime: 121.6907
eval_samples_per_second: 205.4390
eval_steps_per_second: 6.4260
epoch: 5.0000
Testing Predictions on New Samples

Text: This movie was absolutely fantastic! I loved every minute of it.
Prediction: Positive (Confidence: 98.40%)
Probabilities: Negative=0.0160, Positive=0.9840

Text: Terrible film. Complete waste of time and money.
Prediction: Negative (Confidence: 99.09%)
Probabilities: Negative=0.9909, Positive=0.0091

Text: It was okay, nothing special but not bad either.
Prediction: Negative (Confidence: 84.61%)
Probabilities: Negative=0.8461, Positive=0.1539
Fine-tuning Complete!
